### 1.1.12. Sets of Longer Tuples

Section 1.1.10 introduced sets whose members are **ordered pairs** (2-tuples). This section extends that foundation to **triples** (3-tuples) and sets of any dimension $n \geq 3$. The same structural ideas — declaration, `within` constraints, `cross` products, indexing, slicing, and `setof` — all carry over unchanged; only the number of components in each tuple increases.

The canonical motivating model is the **multicommodity transportation problem**: goods flow from origins to destinations, but each origin-destination lane carries multiple products. A natural index set is therefore a set of *(origin, destination, product)* triples.

**Topics covered:**
- Declaring sets of triples: `dimen 3`, `within`, and full Cartesian product forms
- How `cross` chains to produce sets of any dimension
- Indexing over $n$-dimensional sets: `{(i,j,k) in S}`
- Slices: fixing some components and ranging over the rest
- Data supply in `.dat` files for 3-D and higher-dimensional sets
- Parameters indexed over 3-D sets
- The `setof` operator for projecting to lower-dimensional sets
- Mixing dimensions: `LINKS cross PROD` where `LINKS` is already 2-D
- amplpy: assigning $n$-D sets from lists of tuples

In [ ]:
# ── Formal syntax: n-dimensional set declarations ──────────────────────────
#
# set name dimen n ;                         # any n-tuple; component sets not specified
# set name within S1 cross S2 cross ... Sn ; # n-tuple subset with membership constraint
# set name = S1 cross S2 cross ... Sn ;      # full Cartesian product (symbolic definition)
#
# Combined attributes (order-independent):
# set name dimen n within S1 cross S2 cross ... Sn ;
#
# Indexing over an n-D set:
# {(i, j, k)       in S3}   # destructure triple into named dummy indices
# {(i, j, k, l)    in S4}   # destructure 4-tuple
# {(i, j, k) in S3: cond}   # with membership filter
#
# Slice (some components pre-bound):
# {(i0, j, k) in S3}        # i0 already defined; j and k range over matching members
#
# setof operator:
# setof {(i,j,p) in S3} (i,j)    # project to 2-D set of (i,j) pairs
# setof {(i,j,p) in S3} p        # project to 1-D set of p values
# setof {(i,j,p) in S3: cond} p  # conditional projection

#### Declaring Sets of Triples

A set of **triples** is a finite collection of ordered 3-tuples $(a, b, c)$ where every member has exactly three components. AMPL enforces dimensional uniformity: all members of a set must have the same dimension. A set cannot contain both pairs and triples.

There are three declaration forms, differing in how much structural information is supplied in the model:

| Form | Syntax | Meaning |
|---|---|---|
| Bare dimension | `set ROUTES dimen 3;` | Members are 3-tuples; no constraint on components |
| Subset with `within` | `set ROUTES within ORIG cross DEST cross PROD;` | Members are 3-tuples; each component must come from the named 1-D set |
| Full Cartesian product | `set ROUTES = ORIG cross DEST cross PROD;` | Symbolically defined as all possible triples from the three sets |

The `dimen n` phrase tells AMPL the dimension of the set without specifying which sets the components come from. The `within` phrase both specifies dimension (as the dimension of the right-hand side expression) and enforces a membership constraint at data-load time: AMPL checks that each supplied tuple has its first component in `ORIG`, its second in `DEST`, and its third in `PROD`.

**Key property of `within` on high-dimensional sets.** When you write `set ROUTES within ORIG cross DEST cross PROD`, AMPL does *not* materialize the full Cartesian product `ORIG cross DEST cross PROD` in memory. It merely uses the component sets to validate incoming data tuples. With 10 origins, 100 destinations, and 100 products the full product has 100 000 members, but AMPL only needs to check each supplied triple against the three 1-D sets. This makes `within` declarations both safe and efficient for large sparse sets.

In [ ]:
# Three equivalent-looking but semantically distinct declarations

set ORIG;   # 1-D set of origins
set DEST;   # 1-D set of destinations
set PROD;   # 1-D set of products

# Form 1: bare dimension — AMPL only knows tuples are 3-D; no component checking
set ROUTES1 dimen 3;

# Form 2: subset constraint — AMPL validates each triple against ORIG, DEST, PROD
set ROUTES2 within ORIG cross DEST cross PROD;

# Equivalent to Form 2 using indexing-expression syntax (Fourer et al. preferred form)
set ROUTES3 within {ORIG, DEST, PROD};

# Form 3: full Cartesian product (symbolic definition) — ALL (orig,dest,prod) triples
set ROUTES_ALL = ORIG cross DEST cross PROD;

#### How `cross` Chains to Higher Dimensions

The `cross` operator is associative: `A cross B cross C` is evaluated left-to-right as `(A cross B) cross C`, but the result is the same 3-D set regardless of grouping. More generally:

$$A_1 \times A_2 \times \cdots \times A_n = \{(a_1, a_2, \ldots, a_n) : a_k \in A_k \text{ for each } k\}$$

**Dimension arithmetic.** When both operands are 1-D, `A cross B` is 2-D. When the left operand is $d_1$-dimensional and the right is $d_2$-dimensional, `L cross R` is $(d_1 + d_2)$-dimensional. This means that if `LINKS` is a 2-D set of origin-destination pairs, then `LINKS cross PROD` is already a 3-D set of *(origin, destination, product)* triples — even though only two operands appear:

```
LINKS                 dimen 2   e.g. (GARY, DET)
PROD                  dimen 1   e.g. bands
LINKS cross PROD      dimen 3   e.g. (GARY, DET, bands)
```

AMPL "flattens" the tuple: a member of `LINKS cross PROD` is a single 3-tuple, not a nested pair-within-pair.

In [ ]:
set ORIG;  set DEST;  set PROD;

set LINKS within ORIG cross DEST;   # 2-D set of (orig, dest) pairs

# Chaining cross: three 1-D sets → 3-D set
set FULL3D = ORIG cross DEST cross PROD;   # all (orig, dest, prod) triples

# Mixing dimensions: 2-D set cross 1-D set → 3-D set
# Members are (orig, dest, prod) triples, but LINKS need not be ORIG cross DEST
set SPARSE3D = LINKS cross PROD;   # only triples for links that exist

# 4-D example: (orig, dest, prod, week)
set WEEKS;
set SCHED = ORIG cross DEST cross PROD cross WEEKS;  # 4-D set; members are 4-tuples

# A sparse 4-D set: valid only for certain route-week combinations
set TRIPS within LINKS cross PROD cross WEEKS;   # dimen inferred as 2+1+1 = 4

#### Indexing Over $n$-Dimensional Sets

To iterate over an $n$-D set, you destructure each tuple into $n$ named dummy indices by writing the tuple in parentheses to the left of `in`:

```
{(i, j, p) in ROUTES}    # i = origin, j = dest, p = product
{(i, j, p, t) in TRIPS}  # 4-tuple
```

Each dummy index introduced this way has a scope that extends through the rest of the indexing expression and the body of the statement (constraint, objective, sum, etc.).

**Slice indexing.** When some dummy indices are *already defined* (bound in an outer indexing expression), writing them as fixed components produces a **slice**: AMPL iterates only over tuples in the set whose fixed components match the pre-bound values. This is the primary mechanism for writing constraints that aggregate over a subset of dimensions:

```ampl
subject to Supply {i in ORIG, p in PROD}:
    sum {(i,j,p) in ROUTES} Trans[i,j,p] = supply[i,p];
```

Here `i` and `p` are bound by the outer `{i in ORIG, p in PROD}` indexing. Inside the sum, `{(i,j,p) in ROUTES}` is a slice: `j` is the only free dummy index, ranging over all destinations for which `(i,j,p)` is a member of `ROUTES`. AMPL exploits this structure internally for efficient evaluation — see the pitfall alert below.

#### Example 1 — Minimal 3-D Set: Supply and Demand Over Triples

In [ ]:
# Minimal model: 3-D set with parameter and variable

set ORIG;                              # origins
set DEST;                              # destinations
set PROD;                              # products

set ROUTES within ORIG cross DEST cross PROD;   # sparse 3-D subset

param cost {ROUTES} >= 0;              # shipping cost per unit for each triple
var Trans {ROUTES} >= 0;               # units shipped on each (orig, dest, prod) triple

# Sum over all triples — three dummy indices
minimize Total_Cost:
    sum {(i,j,p) in ROUTES} cost[i,j,p] * Trans[i,j,p];

#### Example 2 — Full Multicommodity Transportation Model

This is the canonical model from Fourer, Gay, and Kernighan (AMPL Book, Chapter 6). Supply constraints sum over destinations $j$; demand constraints sum over origins $i$. Each summation is a **slice** through `ROUTES` in which two of the three indices are pre-bound.

In [ ]:
# Multicommodity transportation model — set of triples with slice constraints

set ORIG;                                       # supply cities
set DEST;                                       # demand cities
set PROD;                                       # products

set ROUTES within {ORIG, DEST, PROD};           # feasible (orig, dest, prod) combinations

param supply {ORIG, PROD} >= 0;                 # supply of each product at each origin
param demand {DEST, PROD} >= 0;                 # demand for each product at each destination
param limit  {ORIG, DEST} >= 0;                 # combined capacity of each (orig, dest) lane
param cost   {ROUTES}     >= 0;                 # unit shipping cost per triple

var Trans {ROUTES} >= 0;                        # units shipped on each triple

minimize Total_Cost:
    sum {(i,j,p) in ROUTES} cost[i,j,p] * Trans[i,j,p];

# Supply constraint: i and p are bound; j is free — slice over all destinations
subject to Supply {i in ORIG, p in PROD}:
    sum {(i,j,p) in ROUTES} Trans[i,j,p] = supply[i,p];

# Demand constraint: j and p are bound; i is free — slice over all origins
subject to Demand {j in DEST, p in PROD}:
    sum {(i,j,p) in ROUTES} Trans[i,j,p] = demand[j,p];

# Mixed-load constraint: i and j are bound; p is free — slice over all products
subject to Multi {i in ORIG, j in DEST}:
    sum {(i,j,p) in ROUTES} Trans[i,j,p] <= limit[i,j];

#### Data Supply for 3-D Sets in `.dat` Files

The data for a 3-D set is a list of triples. AMPL accepts three equivalent notations:

1. **Parenthesized triples** — each member written as `(a,b,c)`
2. **Bare triples** — three space-separated tokens per member (no parentheses)
3. **Template (slice) notation** — a pattern with wildcards `*` fixes some components; following tokens supply the remaining ones

All three forms produce identical set membership. The template notation is most compact when the data has structure (e.g., many members share the same origin and product).

In [ ]:
# .dat file data for ROUTES — three equivalent forms

# Form 1: parenthesized triples (explicit)
data;
set ORIG := GARY CLEV PITT ;
set DEST := FRA DET LAN WIN STL FRE LAF ;
set PROD := bands coils ;

set ROUTES :=
    (GARY,LAN,coils)  (GARY,STL,coils)  (GARY,LAF,coils)
    (CLEV,FRA,bands)  (CLEV,FRA,coils)  (CLEV,DET,bands)
    (CLEV,DET,coils)  (CLEV,LAN,bands)  (CLEV,LAN,coils)
    (CLEV,WIN,coils)  (CLEV,STL,bands)  (CLEV,STL,coils)
    (CLEV,LAF,bands)  (PITT,FRA,bands)  (PITT,WIN,bands)
    (PITT,STL,bands)  (PITT,FRE,bands)  (PITT,FRE,coils) ;

# Form 2: bare triples (no parentheses)
set ROUTES :=
    GARY LAN coils   GARY STL coils   GARY LAF coils
    CLEV FRA bands   CLEV FRA coils   CLEV DET bands
    CLEV DET coils   CLEV LAN bands   CLEV LAN coils
    CLEV WIN coils   CLEV STL bands   CLEV STL coils
    CLEV LAF bands   PITT FRA bands   PITT WIN bands
    PITT STL bands   PITT FRE bands   PITT FRE coils ;

# Form 3: template notation — fix first and third components, list destinations
# Template (origin, *, product): following items supply the middle component
set ROUTES :=
    (CLEV,*,bands) FRA DET LAN STL LAF
    (PITT,*,bands) FRA WIN STL FRE
    (GARY,*,coils) LAN STL LAF
    (CLEV,*,coils) FRA DET LAN WIN STL
    (PITT,*,coils) FRE ;

#### The `setof` Operator — Projecting to Lower Dimensions

The `setof` operator constructs a new set by evaluating an expression for each member of an indexing set. It is the standard way to **project** a high-dimensional set onto fewer dimensions:

```
setof { indexing-expression } member-expression
```

The `member-expression` is evaluated once per element of the indexing set; all results are collected into a new set (duplicates are discarded). The dimension of the result equals the dimension of the `member-expression`.

**Common uses:**
- Extract the set of products that appear in at least one route: `setof {(i,j,p) in ROUTES} p`
- Extract the set of (orig, dest) pairs that carry any product: `setof {(i,j,p) in ROUTES} (i,j)`
- Build a filtered version: `setof {(i,j,p) in ROUTES: cost[i,j,p] < 20} (i,j,p)`

This is especially useful when the 1-D component sets (`ORIG`, `DEST`, `PROD`) are not supplied directly in the data, but can be inferred from the higher-dimensional set.

In [ ]:
# setof: derive component sets from a bare 3-D set

# Scenario: only ROUTES (3-D) is supplied in the data; component sets are derived
set ROUTES dimen 3;   # bare declaration — no within constraint

# Project to 1-D: set of all origins that appear in any triple
set ORIG  = setof {(i,j,p) in ROUTES} i;

# Project to 1-D: set of all destinations that appear in any triple
set DEST  = setof {(i,j,p) in ROUTES} j;

# Project to 1-D: set of all products that appear in any triple
set PROD  = setof {(i,j,p) in ROUTES} p;

# Project to 2-D: set of (orig, dest) pairs that carry at least one product
set LINKS = setof {(i,j,p) in ROUTES} (i,j);

# Filtered projection: only (orig, dest) pairs carrying 'bands'
set BANDS_LINKS = setof {(i,j,p) in ROUTES: p = 'bands'} (i,j);

# Projection with a condition on a parameter:
param cost {ROUTES} >= 0;
set CHEAP_ROUTES = setof {(i,j,p) in ROUTES: cost[i,j,p] < 20} (i,j,p);

#### Example 3 — Multicommodity Transportation with Derived Component Sets

This example shows the `dimen 3` + `setof` pattern: only `ROUTES` is supplied in the data. The component sets `ORIG`, `DEST`, and `PROD` are derived, and the constraints use set slices for efficient aggregation.

In [ ]:
# Multicommodity transportation: component sets derived via setof

set ROUTES dimen 3;                              # data supplies triples directly

set ORIG  = setof {(i,j,p) in ROUTES} i;        # all origins that appear in any route
set DEST  = setof {(i,j,p) in ROUTES} j;        # all destinations that appear in any route
set PROD  = setof {(i,j,p) in ROUTES} p;        # all products that appear in any route
set LINKS = setof {(i,j,p) in ROUTES} (i,j);    # (orig, dest) pairs with at least one product

param supply {ORIG, PROD} >= 0;
param demand {DEST, PROD} >= 0;
param limit  {LINKS}      >= 0;                 # per-lane capacity (all products combined)
param cost   {ROUTES}     >= 0;

var Trans {ROUTES} >= 0;

minimize Total_Cost:
    sum {(i,j,p) in ROUTES} cost[i,j,p] * Trans[i,j,p];

subject to Supply {i in ORIG, p in PROD}:
    sum {(i,j,p) in ROUTES} Trans[i,j,p] = supply[i,p];

subject to Demand {j in DEST, p in PROD}:
    sum {(i,j,p) in ROUTES} Trans[i,j,p] = demand[j,p];

subject to Multi {(i,j) in LINKS}:
    sum {(i,j,p) in ROUTES} Trans[i,j,p] <= limit[i,j];

# Corresponding .dat data (no ORIG/DEST/PROD needed; only ROUTES and parameters):
data;
set ROUTES :=
    GARY LAN coils   GARY STL coils   GARY LAF coils
    CLEV FRA bands   CLEV DET bands   CLEV LAN bands
    CLEV FRA coils   CLEV DET coils   CLEV LAN coils
    PITT FRA bands   PITT STL bands   PITT FRE bands ;

#### Slices of 3-D Sets in Detail

A **slice** of a set $S$ of dimension $n$ is obtained by fixing the values of $k < n$ components and letting the remaining $n - k$ components range freely over all matching members. In AMPL, a slice is expressed by placing already-defined indices (or explicit values) in the fixed positions of a destructuring pattern.

For a 3-D set `ROUTES` of *(orig, dest, prod)* triples, the possible slices are:

| Fixed components | Free components | Indexing syntax | Result dimension |
|---|---|---|---|
| origin `i` and product `p` | destination `j` | `{(i,j,p) in ROUTES}` | 1-D (over `j`) |
| origin `i` and destination `j` | product `p` | `{(i,j,p) in ROUTES}` | 1-D (over `p`) |
| destination `j` and product `p` | origin `i` | `{(i,j,p) in ROUTES}` | 1-D (over `i`) |
| origin `i` only | `(j, p)` | `{(i,j,p) in ROUTES}` | 2-D (over `j` and `p`) |

The same syntax `{(i,j,p) in ROUTES}` covers all cases: AMPL determines at model-generation time which indices are already bound and which are free.

**Efficiency note.** The slice form `{(i,j,p) in ROUTES}` — with `i` and `p` pre-bound — allows AMPL to retrieve directly all triples in `ROUTES` with given first and third components. The explicit membership-test form `{j in DEST, p in PROD: (i,j,p) in ROUTES}` forces AMPL to generate the full set `DEST cross PROD` and test each pair, which is far less efficient for large sets.

In [ ]:
# Illustrating all slice forms for a 3-D set

set ORIG;  set DEST;  set PROD;
set ROUTES within ORIG cross DEST cross PROD;
var Trans {ROUTES} >= 0;
param supply_all {ORIG} >= 0;   # total capacity for all products out of each origin

# Slice 1: fix i and p — sum over j (all destinations for this origin-product pair)
param supply {ORIG, PROD} >= 0;
subject to Supply {i in ORIG, p in PROD}:
    sum {(i,j,p) in ROUTES} Trans[i,j,p] = supply[i,p];

# Slice 2: fix j and p — sum over i (all origins serving this destination-product pair)
param demand {DEST, PROD} >= 0;
subject to Demand {j in DEST, p in PROD}:
    sum {(i,j,p) in ROUTES} Trans[i,j,p] = demand[j,p];

# Slice 3: fix i and j — sum over p (all products on this lane)
param limit {ORIG, DEST} >= 0;
subject to Multi {i in ORIG, j in DEST}:
    sum {(i,j,p) in ROUTES} Trans[i,j,p] <= limit[i,j];

# Slice 4: fix i only — sum over both j and p (all outbound from this origin)
# PREFERRED form: compact, efficient
subject to Supply_All {i in ORIG}:
    sum {(i,j,p) in ROUTES} Trans[i,j,p] <= supply_all[i];

# LESS EFFICIENT equivalent (avoids slice notation — forces explicit membership test)
# subject to Supply_All_Slow {i in ORIG}:
#     sum {j in DEST, p in PROD: (i,j,p) in ROUTES} Trans[i,j,p] <= supply_all[i];

<div class="alert alert-block alert-success"> <b>Best practice: </b><br>
Declare 3-D (and higher-dimensional) sets with <code>within S1 cross S2 cross S3</code> rather than bare <code>dimen 3</code> whenever the component sets are available in the model. The <code>within</code> phrase causes AMPL to validate each incoming tuple at data-load time, producing a specific error message ("element not in set") if a component is misspelled or out of range. Without <code>within</code>, a misspelled origin city silently creates a new, unexpected member in the set.</div>

<div class="alert alert-block alert-success"> <b>Best practice: </b><br>
Use slice notation <code>{(i,j,p) in ROUTES}</code> rather than the explicit membership-test form <code>{j in DEST, p in PROD: (i,j,p) in ROUTES}</code> when writing constraints that aggregate over a subset of components. The slice form allows AMPL to use an efficient internal lookup rather than generating and testing the full Cartesian product. For problems with hundreds of destinations and products this difference can be the deciding factor between tractable and intractable model generation.</div>

<div class="alert alert-block alert-warning"> <b>Pitfall: </b><br>
All tuples in a set must have the same dimension. AMPL enforces this as a compile-time rule: a set cannot contain both pairs and triples, and the dimension of a set cannot be determined by data values. If your model sometimes requires 2-component keys and sometimes 3-component keys, use two separate sets of the appropriate dimensions rather than trying to mix them in one set.</div>

<div class="alert alert-block alert-warning"> <b>Pitfall: </b><br>
The full Cartesian product <code>ORIG cross DEST cross PROD</code> can be very large. With 10 origins, 100 destinations, and 100 products it has 100 000 members. The <code>within</code> form of the declaration avoids materializing this product; AMPL only uses the component sets for validation. By contrast, writing <code>set ROUTES = ORIG cross DEST cross PROD</code> (with <code>=</code>) does define the set to be the full product, which may consume substantial memory. Use the <code>=</code> form only when you genuinely need all possible combinations.</div>

#### amplpy: Assigning $n$-D Sets from Lists of Tuples

In amplpy, a multi-dimensional set is assigned by passing a list of Python tuples, one tuple per set member. Each tuple must have exactly $n$ elements matching the declared dimension of the set.

| Pattern | Syntax | Notes |
|---|---|---|
| Property assignment | `ampl.set["ROUTES"] = [("GARY","LAN","coils"), ...]` | Most concise |
| Method call | `ampl.get_set("ROUTES").set_values([("GARY","LAN","coils"), ...])` | Explicit |
| Read back | `ampl.get_set("ROUTES").members()` | Returns list of tuples |
| Via eval | `ampl.eval("set ROUTES := (GARY,LAN,coils) ...;")` | Raw AMPL string |

When you read a 3-D set back with `members()`, amplpy returns a Python list of 3-tuples. Reading via `get_values()` returns an amplpy DataFrame whose columns correspond to the positional dimensions.

In [ ]:
from amplpy import AMPL

ampl = AMPL()

# Declare the model
ampl.eval("""
    set ORIG;  set DEST;  set PROD;
    set ROUTES within ORIG cross DEST cross PROD;
    param cost {ROUTES} >= 0;
""")

# Assign component sets
ampl.set["ORIG"] = ["GARY", "CLEV", "PITT"]
ampl.set["DEST"] = ["FRA", "DET", "LAN", "WIN", "STL", "FRE", "LAF"]
ampl.set["PROD"] = ["bands", "coils"]

# Assign 3-D set from a list of tuples
routes_data = [
    ("GARY", "LAN",  "coils"),
    ("GARY", "STL",  "coils"),
    ("GARY", "LAF",  "coils"),
    ("CLEV", "FRA",  "bands"),
    ("CLEV", "FRA",  "coils"),
    ("CLEV", "DET",  "bands"),
    ("CLEV", "DET",  "coils"),
    ("CLEV", "LAN",  "bands"),
    ("PITT", "FRA",  "bands"),
    ("PITT", "STL",  "bands"),
    ("PITT", "FRE",  "bands"),
    ("PITT", "FRE",  "coils"),
]

# Pattern 1: property assignment (most concise)
ampl.set["ROUTES"] = routes_data

# Pattern 2: method call (equivalent)
ampl.get_set("ROUTES").set_values(routes_data)

# Assign cost parameter indexed over ROUTES
cost_data = {
    ("GARY", "LAN",  "coils"): 11,
    ("GARY", "STL",  "coils"): 16,
    ("GARY", "LAF",  "coils"): 8,
    ("CLEV", "FRA",  "bands"): 27,
    ("CLEV", "FRA",  "coils"): 23,
    ("CLEV", "DET",  "bands"): 9,
    ("CLEV", "DET",  "coils"): 8,
    ("CLEV", "LAN",  "bands"): 12,
    ("PITT", "FRA",  "bands"): 24,
    ("PITT", "STL",  "bands"): 28,
    ("PITT", "FRE",  "bands"): 99,
    ("PITT", "FRE",  "coils"): 81,
}
ampl.get_parameter("cost").set_values(cost_data)

# Read back: members() returns a list of 3-tuples
members = ampl.get_set("ROUTES").members()
print("ROUTES members:", members[:3], "...")  # first three triples

# Read back as amplpy DataFrame
df = ampl.get_set("ROUTES").get_values()
print(df)

#### amplpy: Building Routes from a Pandas DataFrame

In [ ]:
import pandas as pd
from amplpy import AMPL

ampl = AMPL()
ampl.eval("""
    set ORIG;  set DEST;  set PROD;
    set ROUTES within ORIG cross DEST cross PROD;
    param cost {ROUTES} >= 0;
""")

# Build data from a pandas DataFrame (e.g., read from CSV or database)
df = pd.DataFrame({
    "orig":  ["GARY", "GARY", "CLEV", "CLEV", "PITT", "PITT"],
    "dest":  ["LAN",  "STL",  "FRA",  "DET",  "FRA",  "FRE"],
    "prod":  ["coils","coils","bands","bands","bands","coils"],
    "cost":  [11,     16,     27,     9,      24,     81    ],
})

# Extract component sets from the DataFrame columns
ampl.set["ORIG"] = df["orig"].unique().tolist()
ampl.set["DEST"] = df["dest"].unique().tolist()
ampl.set["PROD"] = df["prod"].unique().tolist()

# Extract 3-D set membership as list of tuples
routes_tuples = list(df[["orig","dest","prod"]].itertuples(index=False, name=None))
ampl.set["ROUTES"] = routes_tuples

# Assign cost parameter from the DataFrame (index = tuple key)
cost_dict = {(r.orig, r.dest, r.prod): r.cost for r in df.itertuples(index=False)}
ampl.get_parameter("cost").set_values(cost_dict)

print("Loaded", ampl.get_set("ROUTES").size(), "routes.")

#### Edge Cases

**Edge case 1: Empty 3-D set — constraints generate zero instances.**

A 3-D set may be empty for a particular data instance. All constraints indexed over that set generate zero instances — this is well-defined and not an error. Aggregation expressions (`sum`, `min`, `max`) over an empty index set return their identity values: `sum` returns 0, and `min`/`max` produce errors if no members exist. A `within` constraint on an empty set is satisfied trivially.

In [ ]:
# Edge case 1: empty ROUTES — zero constraints generated; model remains valid

set ORIG := GARY CLEV PITT;
set DEST := FRA DET LAN;
set PROD := bands coils;
set ROUTES within ORIG cross DEST cross PROD;

param cost {ROUTES} >= 0;
var Trans {ROUTES} >= 0;

# When ROUTES is empty:
# - Total_Cost = 0 (sum over empty set)
# - Supply, Demand, Multi generate no constraint instances
# - card(ROUTES) = 0
# - The model is solvable (trivially feasible with zero variables)

minimize Total_Cost:
    sum {(i,j,p) in ROUTES} cost[i,j,p] * Trans[i,j,p];

data;
set ROUTES :=  ;   # empty — no triples supplied

**Edge case 2: `dimen 3` with `setof` — component sets are derived, not declared.**

When only `ROUTES dimen 3` is declared, AMPL cannot validate incoming tuples against component sets (no `within` constraint exists). The component sets `ORIG`, `DEST`, `PROD` must either be declared separately and supplied in data, or derived via `setof` after `ROUTES` is loaded. Derived sets update automatically if `ROUTES` is changed via `let`.

In [ ]:
# Edge case 2: bare dimen 3 — no component validation at data load time

set ROUTES dimen 3;   # bare: any 3-tuple accepted without component checking

# These are derived symbolically — they update if ROUTES changes
set ORIG  = setof {(i,j,p) in ROUTES} i;
set DEST  = setof {(i,j,p) in ROUTES} j;
set PROD  = setof {(i,j,p) in ROUTES} p;

# If a typo appears in data (e.g., 'GARRY' instead of 'GARY'),
# AMPL will silently add 'GARRY' to ORIG — no error is raised.
# Using 'within ORIG cross DEST cross PROD' would have caught this.

data;
set ROUTES := GARY LAN coils   GARY STL coils   CLEV FRA bands ;

display ORIG;   # {GARY, CLEV}
display DEST;   # {LAN, STL, FRA}
display PROD;   # {coils, bands}

**Edge case 3: Mixing `LINKS cross PROD` vs. `ROUTES within ORIG cross DEST cross PROD` — the membership difference.**

`LINKS cross PROD` contains exactly the triples `(i,j,p)` where `(i,j)` is in `LINKS` and `p` is in `PROD`. This is a 3-D set derived from a 2-D set and a 1-D set. Its membership is the *full* Cartesian product of `LINKS` with `PROD`: every product is available on every link. By contrast, `ROUTES within ORIG cross DEST cross PROD` is a *sparse* subset: only specific *(orig, dest, prod)* combinations are members, and the data controls which ones.

In [ ]:
# Edge case 3: LINKS cross PROD (rectangular) vs. ROUTES (sparse)

set ORIG;  set DEST;  set PROD;
set LINKS within ORIG cross DEST;   # sparse set of (orig, dest) pairs

# Every product is available on every existing link — rectangular 3-D set
param cost_rect {LINKS, PROD} >= 0;   # indexed over {LINKS, PROD} — same as LINKS cross PROD
var Trans_rect  {LINKS, PROD} >= 0;

# Only specific (orig, dest, prod) combinations exist — sparse 3-D set
set ROUTES within ORIG cross DEST cross PROD;
param cost_sparse {ROUTES} >= 0;
var Trans_sparse  {ROUTES} >= 0;

# Key difference:
# If LINKS has 10 members and PROD has 3 members, LINKS cross PROD has 30 members.
# ROUTES may have fewer (e.g., 20) if not all products travel all links.
#
# {(i,j) in LINKS, p in PROD} and {LINKS, PROD} are equivalent indexing expressions;
# both denote the full product. ROUTES is a strict subset of this product.

# Constraints look identical in structure but iterate over different sets:
minimize Rect_Cost:   sum {(i,j) in LINKS, p in PROD} cost_rect[i,j,p] * Trans_rect[i,j,p];
minimize Sparse_Cost: sum {(i,j,p) in ROUTES}         cost_sparse[i,j,p] * Trans_sparse[i,j,p];

**Edge case 4: 4-tuples and general $n$-tuples — the pattern extends uniformly.**

The triple mechanism extends without change to 4-tuples, 5-tuples, and higher. A 4-D set needs four dummy indices in destructuring; its `within` constraint is a 4-way `cross` product; its slice indexing works the same way. The data syntax generalizes: bare quadruples are four space-separated tokens; template notation with four entries (some wildcards) works for slices.

In [ ]:
# Edge case 4: 4-D set — schedule of (orig, dest, prod, week) combinations

set ORIG;  set DEST;  set PROD;
set WEEKS ordered;   # planning horizon

# Sparse 4-D set: only certain (orig, dest, prod, week) combinations are active
set SCHED within ORIG cross DEST cross PROD cross WEEKS;

param vol    {SCHED} >= 0;   # scheduled volume
param cost4  {SCHED} >= 0;   # cost per unit for this schedule entry
var   Ship4  {SCHED} >= 0, <= vol;

minimize Total4:
    sum {(i,j,p,t) in SCHED} cost4[i,j,p,t] * Ship4[i,j,p,t];

# Constraint: total shipment for a given (orig, dest, prod) in each week
param weekly_cap {ORIG, DEST, WEEKS} >= 0;
subject to Weekly {i in ORIG, j in DEST, t in WEEKS}:
    sum {(i,j,p,t) in SCHED} Ship4[i,j,p,t] <= weekly_cap[i,j,t];

# Data syntax: bare 4-tuples or template with one or more wildcards
data;
set ORIG  := GARY CLEV ;
set DEST  := FRA DET ;
set PROD  := bands coils ;
set WEEKS := 1 2 3 ;

# Bare quadruples:
set SCHED :=
    GARY FRA bands 1   GARY FRA bands 2
    CLEV DET coils 1   CLEV DET coils 3 ;

# Template notation fixing origin and week:
# (GARY,*,*,1) FRA bands   FRA coils   ...   (one pair per entry)

#### Cross-References

- See also: [1.1.1. Set Declaration vs. Definition](<1_1_1 Set Declaration vs. Definition.ipynb>) — set declaration syntax (`=`, `default`, `within`)
- See also: [1.1.4. Set Operations](<1_1_4 set_operations.ipynb>) — `cross`, `union`, `inter`, `diff`, `setof`
- See also: [1.1.5. Unordered Sets](<1_1_5 unordered_sets.ipynb>) — compound sets are always unordered
- See also: [1.1.2. Indexing Expressions](<1_1_2 Indexing expressions.ipynb>) — scope rules for dummy indices, slice semantics

**Source references:**
- Fourer, Gay, Kernighan: *AMPL: A Modeling Language for Mathematical Programming*, Chapter 6 (Compound Sets and Indexing), Section 6.3 (Sets of Longer Tuples) and Section 6.4 (Operations on Sets of Tuples)
- AMPL Reference Manual, Appendix A.5 (Set Declarations), A.6 (Set Functions)